# UpLift — Notebook 04: SHAP Explainability
**Project:** UpLift — Predictive Maintenance for Transit Accessibility Equipment  
**Author:** Nico Meyering, MPA | Equitech Futures CTI 2026  
**GitHub:** [github.com/meyeringn/uplift-transit](https://github.com/meyeringn/uplift-transit)

---

## What This Notebook Does

Transit agencies cannot act on a black box. A maintenance scheduler who receives a ranked list of at-risk equipment needs to know *why* each unit was flagged — is it age? A pattern of prior failures? An upcoming cold snap?

This notebook uses **SHAP** (SHapley Additive exPlanations) to open that black box. SHAP assigns each feature a contribution value for every individual prediction, grounded in game theory. Think of it like this: if the model's prediction is a pie, SHAP tells you exactly how much of that pie each ingredient contributed — for every single unit, every single time.

**Outputs:**
- SHAP beeswarm summary plot (global feature importance)
- SHAP bar plot (mean absolute impact)
- SHAP waterfall plot (single unit explanation — the kind you'd show a maintenance scheduler)
- `uplift_shap_values.csv` — full SHAP value matrix for all holdout records

In [ ]:
# Install dependencies (run once in Google Colab)
# !pip install xgboost imbalanced-learn shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 1. Load Model and Holdout Data

In [ ]:
model = joblib.load('uplift_xgboost_model.pkl')
X_holdout = np.load('X_holdout.npy')
y_holdout = np.load('y_holdout.npy')
feature_cols = joblib.load('uplift_feature_cols.pkl')

X_holdout_df = pd.DataFrame(X_holdout, columns=feature_cols)
print(f'Holdout set: {X_holdout_df.shape}')
print(f'Features: {feature_cols}')

## 2. Compute SHAP Values

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_holdout_df)
expected_value = explainer.expected_value

print(f'SHAP values shape: {shap_values.shape}')
print(f'Expected value (base rate): {expected_value:.4f}')

# Save full SHAP matrix
shap_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_df.to_csv('uplift_shap_values.csv', index=False)
print('Saved: uplift_shap_values.csv')

## 3. Global Summary — Beeswarm Plot

In [ ]:
plt.figure(figsize=(11, 8))
shap.summary_plot(
    shap_values, X_holdout_df,
    feature_names=feature_cols,
    show=False, plot_size=None
)
plt.title(
    'UpLift — SHAP Feature Importance (Beeswarm)\n'
    'Each dot = one equipment unit. Color = feature value (red=high, blue=low).\n'
    'Position on x-axis = impact on failure probability.',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('uplift_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_shap_beeswarm.png')

## 4. Global Summary — Bar Plot (Mean Absolute SHAP)

In [ ]:
mean_shap = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(mean_shap['feature'], mean_shap['mean_abs_shap'],
               color='#1565C0', alpha=0.85, edgecolor='white')
ax.set_xlabel('Mean |SHAP Value| — Average Impact on Failure Probability', fontsize=11)
ax.set_title(
    'UpLift — Feature Importance\nMean Absolute SHAP Values (Holdout Set)',
    fontweight='bold', fontsize=13
)
# Label bars
for bar, val in zip(bars, mean_shap['mean_abs_shap']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('uplift_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_shap_bar.png')

## 5. Single Unit Explanation — Waterfall Plot

This is the chart you'd show a maintenance scheduler. It answers: *why did the model flag this specific elevator?*

In [ ]:
# Find the highest-risk unit in the holdout set
y_prob = model.predict_proba(X_holdout_df)[:, 1]
highest_risk_idx = np.argmax(y_prob)
risk_score = y_prob[highest_risk_idx]
actual_label = y_holdout[highest_risk_idx]

print(f'Highest-risk unit in holdout set:')
print(f'  Index:        {highest_risk_idx}')
print(f'  Risk score:   {risk_score:.4f}')
print(f'  Actual label: {"OUTAGE" if actual_label == 1 else "No outage"}')
print(f'\nUnit features:')
print(X_holdout_df.iloc[highest_risk_idx])

In [ ]:
# Waterfall plot for highest-risk unit
shap_explanation = shap.Explanation(
    values=shap_values[highest_risk_idx],
    base_values=expected_value,
    data=X_holdout_df.iloc[highest_risk_idx].values,
    feature_names=feature_cols
)

plt.figure(figsize=(11, 7))
shap.plots.waterfall(shap_explanation, show=False)
plt.title(
    f'UpLift — Why Was This Unit Flagged?\n'
    f'Risk Score: {risk_score:.3f} | Actual Outcome: {"Outage" if actual_label == 1 else "No Outage"}',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
plt.savefig('uplift_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: uplift_shap_waterfall.png')

## 6. Key Takeaways

| Finding | Operational Implication |
|---|---|
| `unplanned_outages_12mo` is the dominant feature | Historical failure patterns are the strongest signal — agencies should prioritize complete outage logging |
| `equipment_age_years` is consistently high-impact | Age-based replacement schedules should be calibrated against UpLift scores, not fixed intervals |
| `avg_monthly_temp_f` contributes meaningfully | Seasonal pre-maintenance runs before extreme weather windows are justified by the model |
| `inspection_score` shows negative SHAP (higher score = lower risk) | Inspection programs have measurable predictive value — validate and fund them |
| Individual waterfall plots are agency-ready | Every flagged unit can be explained in plain language to a maintenance supervisor |

➡️ **Next:** Notebook 05 — Threshold Sensitivity Analysis